# 05 — Model deployment on web

In [3]:
import os
import json
import time
from dataclasses import dataclass
from typing import Any, Dict

import numpy as np
import streamlit as st

#MODEL_PATH = "./"


## Streamlit part

In [2]:


# Optionnel : joblib pour charger un modèle sklearn
# pip install joblib
try:
    import joblib
except Exception:
    joblib = None


# ----------------------------
# Configuration
# ----------------------------
APP_TITLE = "ML Predictor (Streamlit + Railway)"
DEFAULT_MODEL_PATH = os.getenv("MODEL_PATH", "model.joblib")  # change si besoin


@dataclass
class PredictionResult:
    prediction: Any
    proba: Any | None = None
    latency_ms: int | None = None
    extra: Dict[str, Any] | None = None


# ----------------------------
# Chargement du modèle (une seule fois)
# ----------------------------
@st.cache_resource(show_spinner=True)
def load_model(model_path: str):
    """
    Charge un modèle ML.
    - Exemple: sklearn joblib (.joblib)
    - Si tu as un autre framework (torch, tf), remplace cette fonction.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"Modèle introuvable: {model_path}. "
            f"Dépose-le dans le repo ou configure MODEL_PATH."
        )

    if joblib is None:
        raise RuntimeError(
            "joblib n'est pas installé. Ajoute `joblib` à requirements.txt "
            "ou adapte load_model() à ton framework."
        )

    model = joblib.load(model_path)
    return model


def predict(model, features: np.ndarray) -> PredictionResult:
    """
    Exécute la prédiction sur un tableau numpy (shape: [n_samples, n_features]).
    """
    t0 = time.time()

    y_pred = model.predict(features)

    proba = None
    if hasattr(model, "predict_proba"):
        try:
            proba = model.predict_proba(features)
        except Exception:
            proba = None

    latency_ms = int((time.time() - t0) * 1000)

    # Pour l'UI, on renvoie le premier échantillon si n_samples==1
    pred_out = y_pred[0] if len(y_pred) == 1 else y_pred
    proba_out = None
    if proba is not None:
        proba_out = proba[0].tolist() if len(proba) == 1 else proba.tolist()

    return PredictionResult(
        prediction=pred_out,
        proba=proba_out,
        latency_ms=latency_ms,
        extra={"n_samples": int(features.shape[0]), "n_features": int(features.shape[1])},
    )


# ----------------------------
# UI Streamlit
# ----------------------------
st.set_page_config(page_title=APP_TITLE, layout="centered")
st.title(APP_TITLE)
st.caption("Démo: charge un modèle, saisit des features, obtient une prédiction.")

with st.sidebar:
    st.header("⚙️ Configuration")
    model_path = st.text_input("Chemin du modèle", value=DEFAULT_MODEL_PATH)
    st.write("Variable d’env possible :", "`MODEL_PATH`")

# Chargement modèle
try:
    model = load_model(model_path)
    st.success(f"Modèle chargé ✅ ({model_path})")
except Exception as e:
    st.error(f"Impossible de charger le modèle : {e}")
    st.stop()

st.subheader("🔢 Entrée des features")

mode = st.radio(
    "Mode d'entrée",
    ["1 ligne (manuel)", "JSON (batch)"],
    horizontal=True,
)

if mode == "1 ligne (manuel)":
    st.write(
        "Entrez les features séparées par des virgules.\n\n"
        "Exemple : `0.12, 3.4, -1.0, 7.8`"
    )
    raw = st.text_input("Features (CSV)", value="")

    col1, col2 = st.columns(2)
    with col1:
        do_predict = st.button("Prédire", type="primary")
    with col2:
        st.button("Réinitialiser", on_click=lambda: None)

    if do_predict:
        try:
            values = [float(x.strip()) for x in raw.split(",") if x.strip() != ""]
            if not values:
                st.warning("Aucune feature saisie.")
                st.stop()

            X = np.array(values, dtype=float).reshape(1, -1)
            result = predict(model, X)

            st.subheader("✅ Résultat")
            st.json(
                {
                    "prediction": result.prediction,
                    "proba": result.proba,
                    "latency_ms": result.latency_ms,
                    "meta": result.extra,
                }
            )
        except Exception as e:
            st.error(f"Erreur pendant la prédiction : {e}")

else:
    st.write(
        "Collez un JSON de type liste de listes (batch).\n\n"
        "Exemple : `[[0.1, 3.4, -1.0], [0.2, 3.5, -0.9]]`"
    )
    raw_json = st.text_area("Features JSON", height=140, value="[]")
    do_predict = st.button("Prédire (batch)", type="primary")

    if do_predict:
        try:
            data = json.loads(raw_json)
            X = np.array(data, dtype=float)
            if X.ndim != 2 or X.shape[0] < 1 or X.shape[1] < 1:
                raise ValueError("Le JSON doit être une liste de listes 2D non vide.")

            result = predict(model, X)

            st.subheader("✅ Résultat")
            st.json(
                {
                    "prediction": result.prediction,
                    "proba": result.proba,
                    "latency_ms": result.latency_ms,
                    "meta": result.extra,
                }
            )
        except Exception as e:
            st.error(f"Erreur pendant la prédiction : {e}")

st.divider()
st.caption("Astuce : pour une vraie app, ajoute validation, logs, monitoring, et un modèle versionné.")


2026-01-21 15:19:34.365 
  command:

    streamlit run C:\apps\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-01-21 15:19:34.368 Session state does not function when running a script without `streamlit run`


DeltaGenerator()

## Exemple: explication d'un modèle déjà en mémoire

Dans une soutenance, vous pouvez :
- recharger le modèle depuis MLflow (Model Registry / artifacts)
- ou réentraîner rapidement le meilleur pipeline

Ici, on montre le principe sur un modèle 'placeholder'.

In [ ]:
# TODO: remplacez par votre meilleur modèle (pipeline) issu du notebook 03
# best_model = ...

# X_sample = X.sample(500, random_state=42)
# explainer = shap.Explainer(best_model, X_sample)
# shap_values = explainer(X_sample)

# shap.plots.bar(shap_values)      # importance globale (bar)
# shap.plots.beeswarm(shap_values) # globale (beeswarm)


## Explication locale (un client)

In [ ]:
# idx = X_sample.index[0]
# shap.plots.waterfall(shap_values[0])  # explication locale
